Import Library

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import json
import random

데이터셋 가져오기, data augmentation

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
torch.manual_seed(42)

train_transform = v2.Compose([
    v2.ToImage(),
    v2.RandomCrop(32, padding=4),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

# Download training data from open datasets.
training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform,
)

# Download test data from open datasets.
test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform,
)

# Create data loaders.
batch_size = 128
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True,
                             num_workers=2, pin_memory=True,
                             persistent_workers=True, drop_last=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size,
                            num_workers=2, pin_memory=True,
                            persistent_workers=True)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

100%|██████████| 170M/170M [25:37<00:00, 111kB/s] 


Shape of X [N, C, H, W]: torch.Size([128, 3, 32, 32])
Shape of y: torch.Size([128]) torch.int64


In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

ResNet

In [ ]:
class Block(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
      super().__init__()
      self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
      self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
      self.bn1 = nn.BatchNorm2d(out_channels)
      self.bn2 = nn.BatchNorm2d(out_channels)
      self.downsample = downsample

    def forward(self, x):
      identity = x
      out = self.conv1(x)
      out = self.bn1(out)
      out = F.relu(out)
      out = self.conv2(out)
      out = self.bn2(out)
      if self.downsample is not None:
        identity = self.downsample(identity)
      return F.relu(out + identity)

class A(nn.Module): #옵션 A 클래스, downsampling 일어날 때 padding으로 크기 조절
    def __init__(self, in_channels, out_channels):
      super().__init__()
      self.in_channels = in_channels
      self.out_channels = out_channels

    def forward(self, x):
      new_x = x[:, :, ::2, ::2]
      new_x = F.pad(new_x, (0, 0, 0, 0, 0, self.out_channels - self.in_channels))
      return new_x

# class B(nn.Module): downsampling 일어날 때만 projection 하는 것이어서 그냥 downsampling 일어날 때 C 호출하면 됨
#     def __init__(self, in_channels, out_channels, downsample=None):
#       super().__init__()
#       self.in_channels = in_channels
#       self.out_channels = out_channels
#       self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
#       self.bn1 = nn.BatchNorm2d(out_channels)

#     def forward(self, x):
#       if downsample is not None:
#         new_x = self.conv1(x)
#         new_x = self.bn1(new_x)

#       return new_x



class C(nn.Module): #옵션 C, downsampling 여부와 관계없이 항상 conv로 크기 맞춤
    def __init__(self, in_channels, out_channels, stride=1):
      super().__init__()
      self.in_channels = in_channels
      self.out_channels = out_channels
      self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, padding=0, bias=False)
      self.bn1 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
      new_x = self.conv1(x)
      new_x = self.bn1(new_x)
      return new_x

class ResNet(nn.Module):
    def __init__(self,  block_list, class_cnt, option='A'):
      super().__init__()
      self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
      self.bn1 = nn.BatchNorm2d(16)
      self.blocks = nn.ModuleList()
      self.in_channels = 16
      self.option = option

      for idx, layer_cnt in enumerate(block_list):
        self.out_channels = 16 * (2 ** idx)

        if idx == 0:
          stride = 1
        else:
          stride = 2 # 첫 블록 이후에는 feature map 크기 절반씩 줄이기

        self.blocks.append(
            self.make_block(self.out_channels, layer_cnt, stride)
        )

      self.fc = nn.Linear(self.out_channels, class_cnt)

    def make_block(self, out_channels, layer_cnt, stride):
      layers = nn.ModuleList()
      layers.append(Block(self.in_channels, out_channels, stride, self.rescale_identity(out_channels, stride)))
      self.in_channels = out_channels

      for i in range(layer_cnt - 1):
          layers.append(Block(out_channels, out_channels, 1, self.rescale_identity(out_channels, 1)))

      return layers

    def rescale_identity(self, out_channels, stride):
      if self.option == 'A':
        if stride != 1 or self.in_channels != out_channels:
          return A(self.in_channels, out_channels)

      elif self.option == 'B':
        if stride != 1 or self.in_channels != out_channels:
          return C(self.in_channels, out_channels, stride)

      elif self.option == 'C':
        return C(self.in_channels, out_channels, stride)

      return None

    def forward(self, x):
      x = self.conv1(x)
      x = self.bn1(x)
      x = F.relu(x)

      for block in self.blocks:
        for layer in block:
          x = layer(x)

      x = F.adaptive_avg_pool2d(x, 1)
      x = torch.flatten(x, 1)
      x = self.fc(x)

      return x

model = ResNet([3, 3, 3], 10, option='C').to(device)
x = torch.randn(1, 3, 32, 32).to(device)
output = model(x)
print(output.shape)

torch.Size([1, 10])


PreActResNet

In [ ]:
class Block(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
      super().__init__()
      self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
      self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
      self.bn1 = nn.BatchNorm2d(in_channels)
      self.bn2 = nn.BatchNorm2d(out_channels)
      self.downsample = downsample

    def forward(self, x):
      identity = x
      out = self.bn1(x)
      out = F.relu(out)
      out = self.conv1(out)

      out = self.bn2(out)
      out = F.relu(out)
      out = self.conv2(out)

      if self.downsample is not None:
        identity = self.downsample(identity)

      return identity + out

class A(nn.Module): #옵션 A 클래스, downsampling 일어날 때 padding으로 크기 조절
    def __init__(self, in_channels, out_channels):
      super().__init__()
      self.in_channels = in_channels
      self.out_channels = out_channels

    def forward(self, x):
      new_x = x[:, :, ::2, ::2]
      new_x = F.pad(new_x, (0, 0, 0, 0, 0, self.out_channels - self.in_channels))
      return new_x

# class B(nn.Module): downsampling 일어날 때만 projection 하는 것이어서 그냥 downsampling 일어날 때 C 호출하면 됨
#     def __init__(self, in_channels, out_channels, downsample=None):
#       super().__init__()
#       self.in_channels = in_channels
#       self.out_channels = out_channels
#       self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
#       self.bn1 = nn.BatchNorm2d(out_channels)

#     def forward(self, x):
#       if downsample is not None:
#         new_x = self.conv1(x)
#         new_x = self.bn1(new_x)

#       return new_x

class C(nn.Module): #옵션 C, downsampling 여부와 관계없이 항상 conv로 크기 맞춤
    def __init__(self, in_channels, out_channels, stride=1):
      super().__init__()
      self.in_channels = in_channels
      self.out_channels = out_channels
      self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, padding=0, bias=False)
      self.bn1 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
      new_x = self.conv1(x)
      new_x = self.bn1(new_x)
      return new_x

class First_Block(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
      super().__init__()
      self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
      self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
      self.bn2 = nn.BatchNorm2d(out_channels)
      self.downsample = downsample

    def forward(self, x):
      identity = x

      out = self.conv1(x)
      out = self.bn2(out)
      out = F.relu(out)
      out = self.conv2(out)

      if self.downsample is not None:
        identity = self.downsample(identity)

      return identity + out

class PreActResNet(nn.Module):
    def __init__(self, block_list, class_cnt, option='B'): # 논문 기본 downsampling 옵션이 B, block_list에는 각 block에 들어갈 layer 개수
      super().__init__()
      self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
      self.bn1 = nn.BatchNorm2d(16)
      self.blocks = nn.ModuleList()
      self.in_channels = 16 # conv1에서 채널 3 -> 16
      self.option = option

      for idx, layer_cnt in enumerate(block_list):
        self.out_channels = 16 * (2 ** idx)

        if idx == 0:
          stride = 1
        else:
          stride = 2 # 첫 블록 이후에는 feature map 크기 절반씩 줄이기

        self.blocks.append(
          self.make_block(self.out_channels, layer_cnt, stride, first=(idx == 0))
        )

      self.bn_final = nn.BatchNorm2d(self.out_channels)
      self.fc = nn.Linear(self.out_channels, class_cnt)

    def make_block(self, out_channels, layer_cnt, stride, first=False):
      layers = nn.ModuleList()
      if first:
        layers.append(First_Block(self.in_channels, out_channels, stride, self.rescale_identity(out_channels, stride)))
      else:
        layers.append(Block(self.in_channels, out_channels, stride, self.rescale_identity(out_channels, stride)))

      self.in_channels = out_channels

      for i in range(layer_cnt - 1):
          layers.append(Block(out_channels, out_channels, 1, self.rescale_identity(out_channels, 1)))

      return layers

    def rescale_identity(self, out_channels, stride):
      if self.option == 'A':
        if stride != 1 or self.in_channels != out_channels:
          return A(self.in_channels, out_channels)

      elif self.option == 'B':
        if stride != 1 or self.in_channels != out_channels:
          return C(self.in_channels, out_channels, stride)

      elif self.option == 'C':
        return C(self.in_channels, out_channels, stride)

      return None

    def forward(self, x):
      x=self.conv1(x)
      x=self.bn1(x)
      x=F.relu(x)

      for block in self.blocks:
        for layer in block:
          x = layer(x)

      x = F.relu(self.bn_final(x))

      x=F.adaptive_avg_pool2d(x, 1)
      x=torch.flatten(x,1)
      x=self.fc(x)
      return x

model = PreActResNet([3, 3, 3], 10, option='B').to(device)
x = torch.randn(1, 3, 32, 32).to(device)
output=model(x)
print(output.shape)

torch.Size([1, 10])


In [ ]:
class PreActBlock(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()
    if stride != 1 or in_channels != out_channels:
      self.downsample = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels)
      )
    else:
      self.downsample = nn.Identity()

  def forward(self, x):
    y = self.conv1(self.relu(self.bn1(x)))
    y = self.conv2(self.relu(self.bn2(x)))
    return y + self.downsample(x)

class PreActResNet(nn.Module):
  def __init__(self):
    self.blocks = nn.ModuleList([
        self.make_block(), # nn.Seq
        self.make_block(),
    ])

  def forward(self, x):
    x = self.conv1(x)
    for block in self.blocks:
      x = block(x)
    x = self.bn(x)
    x = self.relu(x)
    x = self.avgpool(x)
    x = torch.flatten(x, 1)
    x = self.fc(x)
    return x


DenseNet

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, out_channels = 12): # 채널 입력이 m0, 출력이 k
      super().__init__()
      self.conv1 = nn.Conv2d(in_channels, out_channels * 4, kernel_size=1, stride=1, padding=0, bias=False)
      self.conv2 = nn.Conv2d(out_channels * 4, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
      self.bn1 = nn.BatchNorm2d(in_channels)
      self.bn2 = nn.BatchNorm2d(out_channels * 4)

    def forward(self, x):
      x = self.bn1(x)
      x = F.relu(x)
      x = self.conv1(x)
      x = self.bn2(x)
      x = F.relu(x)
      x = self.conv2(x)
      return x

class DenseBlock(nn.Module):
    def __init__(self, in_channels, layer_cnt, growth_rate = 12):
      super().__init__()
      self.layers = nn.ModuleList()
      for i in range(layer_cnt):
         self.layers.append(DenseLayer(in_channels + i * growth_rate, growth_rate))

    def forward(self, x):
      for layer in self.layers:
        out = layer(x)
        x = torch.cat((x, out), 1)
      return x

class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
      super().__init__()
      self.bn = nn.BatchNorm2d(in_channels)
      self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
      self.avgpool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
      x = self.bn(x)
      x = F.relu(x)
      x = self.conv(x)
      x = self.avgpool(x)
      return x

class DenseNet(nn.Module):
    def __init__(self, block_list, class_cnt, k = 12):
      super().__init__()
      self.cur_channels = 2 * k
      self.conv1 = nn.Conv2d(3, 2 * k, kernel_size=3, stride=1, padding=1, bias=False)
      self.blocks = nn.ModuleList()
      self.transitions = nn.ModuleList()

      for idx, layer_cnt in enumerate(block_list):
        self.blocks.append(self.make_block(layer_cnt, k))

        if idx != len(block_list) - 1: # 마지막 블럭이 아니라면 transition layer도 만들기!
          self.transitions.append(TransitionLayer(self.cur_channels, self.cur_channels // 2))
          self.cur_channels = self.cur_channels // 2

      self.bn = nn.BatchNorm2d(self.cur_channels)
      self.fc = nn.Linear(self.cur_channels, class_cnt)

    def make_block(self, layer_cnt, growth_rate):
      block = DenseBlock(self.cur_channels, layer_cnt, growth_rate)
      self.cur_channels = self.cur_channels + layer_cnt * growth_rate
      return block

    def forward(self, x):
      x = self.conv1(x)

      for idx, block in enumerate(self.blocks):
        x = block(x)
        if idx < len(self.transitions): # 마지막 블럭 아니면 transition layer 통과
          x = self.transitions[idx](x)

      x = self.bn(x)
      x = F.relu(x)
      x = F.adaptive_avg_pool2d(x, 1)
      x = torch.flatten(x, 1)
      x = self.fc(x)
      return x

model = DenseNet([8, 8, 8], 10).to(device)
x = torch.randn(1, 3, 32, 32).to(device)
output=model(x)
print(output.shape)

FractalNet

In [ ]:
class Base(nn.Module): # base layer에선 conv, bn, relu만 해주면 됨
    def __init__(self, in_channels, out_channels):
      super().__init__()
      self.conv = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
      self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
      x = self.conv(x)
      x = self.bn(x)
      x = F.relu(x)
      return x

class Block(nn.Module):
    def __init__(self, in_channels, out_channels, C, p_local=0.15): # C step의 프랙탈 구조를 가지는 block
      super().__init__()
      self.C = C
      self.p_local = p_local
      self.total = 2 ** (C - 1) # 가장 깊은 길이와 같음(가장 긴 경로의 길이)
      self.columns = nn.ModuleList() # column 별로 Base layer들을 연결해서 저장하는 module list들의 list
      for c in range(C):
        num = 2 ** c
        temp = nn.ModuleList() # 해당 column의 base layer들 연결한거
        for i in range(num):
          if i == 0:
            temp.append(Base(in_channels, out_channels))
          else:
            temp.append(Base(out_channels, out_channels))

        self.columns.append(temp)

    def check_remain(self, n):
      flags = (torch.rand(n) >= self.p_local) # True면 살려놓음, flag는 bool type tensor
      if not flags.any(): # 싹 다 false이면 하나 살림
        flags[torch.randint(n, (1,))] = True
      return flags

    def forward(self, x, global_col=None):
      if global_col is not None: # 만약 global drop-path라면 join 싹 다 무시하고 한 column만 살리면 되기 때문에 따로 관리
        global_col = min(global_col, self.C - 1) # 블록마다 C가 다르면 현재 블럭의 column 안넘어가게 조절해주기
        for layer in self.columns[global_col]:
          x = layer(x)
        return x

      X = [x] * self.C # C 개의 column에서 다 x를 입력으로 써야하므로 복사해둠
      depth = [0] * self.C

      for cur_depth in range(1, self.total + 1):
        temp = [] # 각 깊이 별로 join할 대상 담을 리스트
        for c in range(self.C): # 각 column에 대해서
          T = 2 ** (self.C - (c + 1)) # column c의 주기 계산하기
          if cur_depth % T == 0: # 현재 깊이가 주기에 맞으면
            X[c] = self.columns[c][depth[c]](X[c]) # 해당 column의 X를 conv에 통과시킴
            depth[c] += 1 # 그 뒤에 해당 column depth 값++
            temp.append(c) # 그리고 join 대상 리스트에 추가하기(컬럼 인덱스 넣어놓음)

        if self.training and len(temp) > 1: # train 단계일때만 && 만약 입력 1개면 살려야하니깐
          flags = self.check_remain(len(temp))
          remain = []
          for i in range(len(temp)):
            if flags[i]:
              remain.append(temp[i])
        else:
          remain = temp

        joined = sum(X[c] for c in remain) / len(remain)
        for c in temp:
          X[c] = joined # join에 참여한 열들 싹 다 join값으로 업데이트

      return X[0] # 어짜피 모든 열들 값 똑같을꺼라서

class FractalNet(nn.Module):
    def __init__(self, block_list, class_cnt, p_local=0.15, p_global=0.5):
      super().__init__()
      self.conv1 = nn.Conv2d(3, 16, 3, stride=1, padding=1, bias=False)
      self.blocks = nn.ModuleList() # block들 담을 module list, 블럭당 프랙탈 단계는 block_list 파라미터로 받을 것
      in_channels = 16 # RGB 3채널로 들어온거 16채널로 conv1에서 늘려줄꺼라
      self.max_C = max(block_list)
      self.p_global = p_global

      for idx, C in enumerate(block_list):
        out_channels = 16 * (2 ** idx)
        self.blocks.append(Block(in_channels, out_channels, C, p_local))
        in_channels = out_channels

      self.fc = nn.Linear(in_channels, class_cnt)

    def forward(self, x):
      global_col = None
      if self.training and torch.rand(1).item() < self.p_global: # train 단계이고, 0.5 확률로 global drop-path 작동
        global_col = random.randrange(self.max_C) # global drop-path일 때 살릴 column 선택

      x = self.conv1(x)
      for block in self.blocks:
        x = block(x, global_col)
        x = F.max_pool2d(x, 2)
      x = F.adaptive_avg_pool2d(x, 1)
      x = torch.flatten(x, 1)
      x = self.fc(x)
      return x

model = FractalNet([3, 3, 3], 10).to(device)
x = torch.randn(1, 3, 32, 32).to(device)
output=model(x)
print(output.shape)

torch.Size([1, 10])


In [ ]:
class FractalBlock(nn.Module):
  pass

Loss function & Optimizer

In [ ]:
import time
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
scaler = torch.amp.GradScaler()

result = {"train_loss": [], "test_loss": [], "test_acc": [], "lr": []}
best_acc = 0.0

Train setting

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    running_loss, seen = 0.0, 0

    for batch, (X, y) in enumerate(dataloader):
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            pred = model(X)
            loss = loss_fn(pred, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * y.size(0)
        seen += y.size(0)

        if batch % 100 == 0:
            print(f"loss: {loss.item():>7f}  [{(batch+1)*len(X):>5d}/{size:>5d}]")

    return running_loss / seen

Test setting

In [ ]:
def test(dataloader, model, loss_fn):
    global best_acc
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                pred = model(X)
                test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

    if correct > best_acc:
        best_acc = correct
        torch.save(model.state_dict(), "FractalNet.pth")

    return test_loss, correct

scheduler

In [ ]:
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[82, 123], gamma=0.1)

train & test

In [ ]:
epochs = 164
start = time.time()

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    cur_lr = optimizer.param_groups[0]['lr']

    tr_loss = train(train_dataloader, model, loss_fn, optimizer)
    te_loss, te_acc = test(test_dataloader, model, loss_fn)
    scheduler.step()

    result["train_loss"].append(tr_loss)
    result["test_loss"].append(te_loss)
    result["test_acc"].append(te_acc)
    result["lr"].append(cur_lr)

    elapsed = time.time() - start
    eta = elapsed / (t+1) * (epochs - t - 1)
    print(f"[{elapsed/60:.1f}m elapsed | ETA {eta/60:.1f}m | best {best_acc*100:.2f}%]\n")

    with open("FractalNet.json", "w") as f:
        json.dump(result, f)

print(f"Done! best test acc = {best_acc*100:.2f}%")

Epoch 1
-------------------------------
loss: 2.427116  [  128/50000]
loss: 1.889500  [12928/50000]
loss: 1.776251  [25728/50000]
loss: 1.835245  [38528/50000]
Test Error: 
 Accuracy: 34.2%, Avg loss: 1.748467 

[0.7m elapsed | ETA 111.4m | best 34.15%]

Epoch 2
-------------------------------
loss: 1.549223  [  128/50000]
loss: 1.789250  [12928/50000]
loss: 1.646694  [25728/50000]
loss: 1.577459  [38528/50000]
Test Error: 
 Accuracy: 47.3%, Avg loss: 1.435780 

[1.2m elapsed | ETA 94.7m | best 47.30%]

Epoch 3
-------------------------------
loss: 1.695194  [  128/50000]
loss: 1.488053  [12928/50000]
loss: 1.614479  [25728/50000]
loss: 1.191867  [38528/50000]
Test Error: 
 Accuracy: 53.6%, Avg loss: 1.259130 

[1.6m elapsed | ETA 87.8m | best 53.65%]

Epoch 4
-------------------------------
loss: 1.419433  [  128/50000]
loss: 1.160225  [12928/50000]
loss: 1.347093  [25728/50000]
loss: 1.021790  [38528/50000]
Test Error: 
 Accuracy: 60.7%, Avg loss: 1.092532 

[2.1m elapsed | ETA 84.2m